# Notebook 0 — Native Contacts (Q)

Compute **Q**, the fraction of native contacts formed in each frame, using the
Best-Hummer-Eaton (2013, PNAS) smooth contact metric implemented in
`src/native_contacts.py`. Q is a standard, structure-agnostic order parameter
for protein folding: Q ≈ 1 means the native (folded) contact network is fully
formed, Q ≈ 0 means it is absent.

This notebook:
1. Computes Q for every frame of the trajectory directly with MDTraj.
2. Plots its distribution and its trace over time.
3. Uses simple Q thresholds to identify folded / unfolded / transition frames —
   a quick, independent sanity check on the cluster-based state assignments
   used in Notebook 2.
4. Saves `data/<system>/Q_values.csv` (`Frame`, `Time_ps`, `Q`) for reuse as a
   reaction-coordinate axis in later notebooks (e.g. Notebook 5).

**Output:** `data/<system>/Q_values.csv`

In [ ]:
import yaml
import os
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.native_contacts import compute_q

## Select system

In [ ]:
with open("../config/chignolin.yaml") as f:   # change to ww_domain.yaml for WW domain
    cfg = yaml.safe_load(f)

SYSTEM  = cfg["system"]
PDB     = cfg["topology_pdb"]
FILES   = cfg["trajectory_xtc"]
STRIDE  = cfg["stride"]
NATIVE  = cfg.get("native_pdb") or PDB   # defaults to the folded reference PDB

print(f"System: {SYSTEM}")
print(f"PDB:    {PDB}")
print(f"Native: {NATIVE}")
print(f"Stride: {STRIDE}")

## Compute Q per frame

In [ ]:
q, n_contacts = compute_q(FILES, PDB, native_pdb=NATIVE, stride=STRIDE)
print(f"Native contacts: {n_contacts}")
print(f"Frames:          {len(q)}")
print(f"Q range:         [{q.min():.3f}, {q.max():.3f}]")

## Distribution and trace

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(q, bins=50, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Q (fraction of native contacts)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"{SYSTEM} — Q distribution")

axes[1].plot(q, lw=0.5, color="steelblue")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Q")
axes[1].set_title(f"{SYSTEM} — Q vs. frame")

plt.tight_layout()
plt.show()

## Identify folded / unfolded frames from Q

Thresholds are read from the config (`q_folded_threshold`, `q_unfolded_threshold`),
defaulting to the conventional 0.8 / 0.2 split. This is an independent check
on the `folded_clusters` / `unfolded_clusters` assignments used in Notebook 2 —
the two should agree on which regions of the trajectory are folded/unfolded.

In [ ]:
Q_FOLDED   = cfg.get("q_folded_threshold", 0.8)
Q_UNFOLDED = cfg.get("q_unfolded_threshold", 0.2)

folded_frames    = np.where(q >= Q_FOLDED)[0]
unfolded_frames  = np.where(q <= Q_UNFOLDED)[0]
transition_frames = np.where((q > Q_UNFOLDED) & (q < Q_FOLDED))[0]

print(f"Folded     (Q >= {Q_FOLDED}): {len(folded_frames):>7,}  ({len(folded_frames)/len(q):.1%})")
print(f"Unfolded   (Q <= {Q_UNFOLDED}): {len(unfolded_frames):>7,}  ({len(unfolded_frames)/len(q):.1%})")
print(f"Transition ({Q_UNFOLDED} < Q < {Q_FOLDED}): {len(transition_frames):>7,}  ({len(transition_frames)/len(q):.1%})")

## Save Q per frame

In [ ]:
out_dir = cfg.get("q_output_dir") or f"../data/{SYSTEM}"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/Q_values.csv"

df_q = pd.DataFrame({
    "Frame": np.arange(len(q)),
    "Time_ps": np.arange(len(q)) * 200 * STRIDE,
    "Q": q,
})
df_q.to_csv(out_path, index=False)
print(f"Saved: {out_path}")